# Criba de motores TTS — Qwen3-TTS 1.7B Base

`Qwen/Qwen3-TTS-12Hz-1.7B-Base` en modo clonación (referencia + su transcripción), `language="Spanish"`, atención SDPA (sin flash-attn, que no compila en Colab sin esperar mucho). Ojo: su fine-tuning oficial deja la voz fija y ya no acepta referencia; aquí solo se mide el zero-shot.

**GPU:** L4. **Tiempo estimado:** unos 40 min, incluida la instalación.

**Cómo se usa** (es igual en todos los cuadernos de la criba):

1. *Entorno de ejecución → Cambiar tipo de entorno* → **L4**.
2. Ejecuta la **celda 1** (instalación). Cuando termine, *Entorno de ejecución → Reiniciar sesión*.
3. Ejecuta las demás celdas **en orden**, de la 2 a la última. La celda 2 comprueba que estén las
   referencias y las frases antes de cargar ningún modelo: si dice `RESULTADO: REVISAR`, para ahí.
4. La última celda **desconecta la GPU sola** para que no siga gastando unidades.

Si la sesión se corta a medias, vuelve a empezar por la celda 1: lo que ya esté en Drive no se
genera otra vez.

Salida: `fine_tunned_naval/outputs/criba/<variante>/` en Drive, un `.wav` por item (`n01`, `v01`,
`b01`, `k01`…) más `registro.jsonl`, `parametros.json` y `entorno.json`.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
!pip install -q qwen-tts==0.1.1

In [ ]:
# 2. Preparar: código del repo, Drive, frases y referencias
import os, sys, subprocess
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun

print(sys.version.split()[0], "|", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                                  capture_output=True, text=True).stdout.strip())
comun.montar_drive()
ITEMS, SILENCIO_MS = comun.cargar_items(None)
REFS = comun.comprobar(ITEMS)

In [ ]:
# 3. Cargar el modelo
import torch
from qwen_tts import Qwen3TTSModel
comun.guardar_entorno("qwen3tts_1.7b", ["qwen-tts", "torch", "transformers", "accelerate"])
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
modelo = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0",
                                       dtype=DTYPE, attn_implementation="sdpa")
_prompts = {}

def sintetizar(texto, ref):
    if ref["wav"] not in _prompts:
        _prompts[ref["wav"]] = modelo.create_voice_clone_prompt(ref_audio=ref["wav"], ref_text=ref["texto"],
                                                                x_vector_only_mode=False)
    wavs, sr = modelo.generate_voice_clone(text=texto, language="Spanish",
                                           voice_clone_prompt=_prompts[ref["wav"]])
    return wavs[0], sr
print("OK,", DTYPE)

In [ ]:
# 4. Generar
comun.generar("qwen3tts_1.7b", sintetizar, ITEMS, REFS, SILENCIO_MS,
              params={"modelo": "Qwen/Qwen3-TTS-12Hz-1.7B-Base", "language": "Spanish",
                      "dtype": str(DTYPE), "x_vector_only_mode": False})

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
for variante in ['qwen3tts_1.7b']:
    reg = comun.SALIDA / variante / "registro.jsonl"
    if not reg.exists():
        print(f"{variante}: sin registro")
        continue
    filas = [json.loads(l) for l in reg.read_text(encoding="utf-8").splitlines()]
    ok = sum(f["ok"] for f in filas)
    print(f"{variante}: {ok} items correctos, {len(filas) - ok} con error")
comun.desconectar()